# MergeSort e Merge (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_java.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `MergeSort` e `Merge`: gli algoritmi in Java, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/MergeSort_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursiveSum` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Merge Sort e Merge

- **MergeSort**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: il sotto-array $a[l..r]$ ordinato in modo non decrescente. Si chiama `MergeSort(a, 1, n)` nello pseudocodice, `merge_sort(a, 0, n-1)` nel codice.
- **Merge**$(a,l,m,r)$ — *Input*: un array e indici $l\le m<r$ tali che $a[l..m]$ e $a[m+1..r]$ siano ordinati. *Output*: $a[l..r]$ ordinato. Copia le due metà in $L$ e $R$ e riversa in $a$, passo dopo passo, il minore fra i primi due elementi non ancora copiati.

Il contatore `comparisons` conta i confronti fra elementi eseguiti da `Merge`. Sull'array dell'esempio $a=[9,12,3,16,19,11,12]$ si stampano le chiamate di `Merge` nell'ordine di esecuzione (con l'array dopo ciascuna, tabella del capitolo; gli indici sono da 0: aggiungendo 1 si ritrovano quelli del capitolo), poi la traccia di `Merge(a, 0, 1, 3)` su $a=[5,8,2,7]$ (esempio dell'implementazione in C++).

In [2]:
%%writefile MergeSort.java
import java.util.*;

public class MergeSort {
    static long comparisons = 0;   // confronti fra elementi eseguiti da Merge
    static int msCalls = 0;        // chiamate di MergeSort

    // merge(a, l, m, r): fonde a[l..m] e a[m+1..r], entrambi ordinati
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;             // dimensione della parte sinistra
        int n2 = r - m;                 // dimensione della parte destra
        int[] L = new int[n1];          // array ausiliari
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {      // fonde L e R in a[l..r]
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {                // copia il resto di L
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {                // copia il resto di R
            a[k] = R[j];
            j++;
            k++;
        }
    }

    // mergeSort(a, l, r): ordina a[l..r]
    static void mergeSort(int[] a, int l, int r) {
        msCalls++;
        if (l < r) {
            int m = l + (r - l) / 2;    // divide
            mergeSort(a, l, m);         // conquer (metà sinistra)
            mergeSort(a, m + 1, r);     // conquer (metà destra)
            merge(a, l, m, r);          // combine
        }
    }

    // ---- varianti che stampano la traccia ----
    static void mergeSortLog(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSortLog(a, l, m);
            mergeSortLog(a, m + 1, r);
            String left = Arrays.toString(Arrays.copyOfRange(a, l, m + 1));
            String right = Arrays.toString(Arrays.copyOfRange(a, m + 1, r + 1));
            merge(a, l, m, r);
            System.out.println("Merge(a," + l + "," + m + "," + r + ")  " + left + " e " + right + "  ->  " + Arrays.toString(a));
        }
    }

    static void mergeTrace(int[] a, int l, int m, int r) {
        int n1 = m - l + 1, n2 = r - m;
        int[] L = Arrays.copyOfRange(a, l, m + 1), R = Arrays.copyOfRange(a, m + 1, r + 1);
        System.out.println("Merge(a, " + l + ", " + m + ", " + r + ") su a = " + Arrays.toString(a));
        System.out.println("n1=" + n1 + ", n2=" + n2 + ", L=" + Arrays.toString(L) + ", R=" + Arrays.toString(R));
        int i = 0, j = 0, k = l, it = 0;
        while (i < n1 && j < n2) {
            it++;
            String head;
            if (L[i] <= R[j]) {
                head = String.format("%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? Sì => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
                a[k] = L[i];
                i++;
            } else {
                head = String.format("%2d. ciclo 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
                a[k] = R[j];
                j++;
            }
            k++;
            System.out.println(head + ", i=" + i + ", j=" + j + ", k=" + k + "; a=" + Arrays.toString(a));
        }
        while (i < n1) {
            a[k] = L[i];
            System.out.println("    ciclo 2 (resto di L): a[" + k + "] = L[" + i + "] = " + L[i] + "; a=" + Arrays.toString(a));
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            System.out.println("    ciclo 3 (resto di R): a[" + k + "] = R[" + j + "] = " + R[j] + "; a=" + Arrays.toString(a));
            j++;
            k++;
        }
        System.out.println("risultato a = " + Arrays.toString(a) + ", confronti = " + it);
    }

    public static void main(String[] args) {
        int[] a = {9, 12, 3, 16, 19, 11, 12};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("Chiamate di Merge in ordine di esecuzione:");
        mergeSortLog(a.clone(), 0, n - 1);
        comparisons = 0;
        msCalls = 0;
        mergeSort(a, 0, n - 1);
        System.out.println("ordinato = " + Arrays.toString(a) + "   chiamate di MergeSort=" + msCalls + " (2n-1=" + (2 * n - 1) + ")   confronti=" + comparisons);
        System.out.println();
        int[] b = {5, 8, 2, 7};
        mergeTrace(b, 0, 1, 3);
    }
}

Writing MergeSort.java


In [3]:
!javac MergeSort.java && java MergeSort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Chiamate di Merge in ordine di esecuzione:
Merge(a,0,0,1)  [9] e [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] e [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] e [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] e [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] e [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] e [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
ordinato = [3, 9, 11, 12, 12, 16, 19]   chiamate di MergeSort=13 (2n-1=13)   confronti=14

Merge(a, 0, 1, 3) su a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. ciclo 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. ciclo 1: L[0]=5 <= R[1]=7? Sì => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. ciclo 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    ciclo 2 (resto di L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
risultato a = [2, 5, 7, 8], confronti = 3


**Prova tu.** Ordina un altro array: il numero di confronti di `Merge` è al massimo $n\lceil\log_2 n\rceil$ e le chiamate di `MergeSort` sono sempre $2n-1$.

In [4]:
%%writefile ProvaMergeSort.java
import java.util.*;

public class ProvaMergeSort {
    public static void main(String[] args) {
        // Ordina un altro array e conta i confronti (al massimo n*ceil(log2 n)).
        int[] b = {38, 27, 43, 3, 9, 82, 10, 5};
        MergeSort.comparisons = 0;
        MergeSort.msCalls = 0;
        MergeSort.mergeSort(b, 0, b.length - 1);
        System.out.println(Arrays.toString(b) + " " + MergeSort.comparisons + " " + MergeSort.msCalls);
    }
}

Writing ProvaMergeSort.java


In [5]:
!javac ProvaMergeSort.java && java ProvaMergeSort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).